In [1]:
from xgboost import XGBClassifier
from xgboost import plot_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np
import scikitplot as skplt

In [21]:
df = pd.read_csv('train.csv')

df = df.drop(columns=['Cabin', 'Name', 'PassengerId', 'Ticket', 'SibSp', 'Parch'])

imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
imp_mode = SimpleImputer(missing_values=np.nan, strategy='most_frequent')

df['Age'] = imp_mean.fit_transform(df[['Age']]).ravel()
df['Embarked'] = imp_mode.fit_transform(df[['Embarked']]).ravel()

# df['Sex'] = LabelEncoder().fit_transform(df['Sex'])
# df['Pclass'] = LabelEncoder().fit_transform(df['Pclass'])
# df['Embarked'] = LabelEncoder().fit_transform(df['Embarked'])
df = pd.get_dummies(df, columns = ['Sex', 'Pclass', 'Embarked'], dtype='int64')

In [3]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], random_state=42)

In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

xgb_model = XGBClassifier()
cv = KFold(n_splits=5, random_state=42, shuffle=True)
parameters = {'n_estimators' : [50,60,70,80,90,100], 'learning_rate':[0.1,0.2,0.3,0.4],
             'max_depth':[3,4,5,6,7]}
model = GridSearchCV(estimator = xgb_model,
                     param_grid = parameters,
                     cv = cv, verbose = 1,
                     n_jobs = 1, refit = True)
model.fit(X_train, y_train)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'n_estimators': [50, 60, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter c

In [5]:
print("Best Estimator:\n", model.best_estimator_)
print("Best Params:\n", model.best_params_)
print("Best Score:\n", model.best_score_)

Best Estimator:
 XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=70,
              n_jobs=None, num_parallel_tree=None, ...)
Best Params:
 {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 70}
Best Score:
 0.8367523285826508


In [6]:
model = XGBClassifier(learning_rate= 0.2, max_depth=3, n_estimators=70)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [23]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
acc

0.9162011173184358

In [9]:
i=0
max_score = 0
idx = 0
size = 0
test_size = [0.2,0.3]
while(i < 2000):
    for j in test_size:
        X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], random_state=i, test_size=j)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        print(f"{i}번째 {j}크기 예측한 결과: {score}")
        if max_score < score:
            max_score = score
            idx = i
            size = j
    i+=1
        
print(max_score, idx, size)

0번째 0.2크기 예측한 결과: 0.8491620111731844
0번째 0.3크기 예측한 결과: 0.8246268656716418
1번째 0.2크기 예측한 결과: 0.7988826815642458
1번째 0.3크기 예측한 결과: 0.7873134328358209
2번째 0.2크기 예측한 결과: 0.770949720670391
2번째 0.3크기 예측한 결과: 0.7873134328358209
3번째 0.2크기 예측한 결과: 0.7932960893854749
3번째 0.3크기 예측한 결과: 0.7910447761194029
4번째 0.2크기 예측한 결과: 0.8268156424581006
4번째 0.3크기 예측한 결과: 0.8582089552238806
5번째 0.2크기 예측한 결과: 0.8212290502793296
5번째 0.3크기 예측한 결과: 0.8246268656716418
6번째 0.2크기 예측한 결과: 0.8491620111731844
6번째 0.3크기 예측한 결과: 0.832089552238806
7번째 0.2크기 예측한 결과: 0.7932960893854749
7번째 0.3크기 예측한 결과: 0.7873134328358209
8번째 0.2크기 예측한 결과: 0.7932960893854749
8번째 0.3크기 예측한 결과: 0.8283582089552238
9번째 0.2크기 예측한 결과: 0.7877094972067039
9번째 0.3크기 예측한 결과: 0.7947761194029851
10번째 0.2크기 예측한 결과: 0.8324022346368715
10번째 0.3크기 예측한 결과: 0.8208955223880597
11번째 0.2크기 예측한 결과: 0.88268156424581
11번째 0.3크기 예측한 결과: 0.8395522388059702
12번째 0.2크기 예측한 결과: 0.7541899441340782
12번째 0.3크기 예측한 결과: 0.7835820895522388
13번째 0.2크기 예측한 결과: 0.843575418994413

In [22]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], random_state=1677, test_size=0.2)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
acc

0.9162011173184358